# Build a YOLOv7-tiny AI Processor

**Author:** Antoine ([@antowan](https://github.com/antowan))

Build a station package that detects **cars, people and trucks** with YOLOv7-tiny. You’ll export the pretrained weights, compile them for the station GPU, then check the package on a sample video. New to AI Processors? [Follow one frame through the application](00-presentation.ipynb) first.

```mermaid
flowchart LR
    W["Weights"] --> O["ONNX graph"] --> E["TensorRT engine"] --> P["AIP archive"] --> V["Video test"]
```

## 0. Open the project on a development station

Use the station’s **App-Builder (Jupyter)** with internet access for the first export. The commands below target JetPack 6.0, DeepStream 7.0 and TensorRT 8.6 on an Orin NX 16 GB. Run one cell at a time (Shift+Enter); stop at the first error.

**Use a checkout reserved for this exercise.** All three notebooks write to the same `/data/aip/AIPV1EMI0001` and replace its active detector. Don’t use an installed or in-use AIP folder or run two flavor notebooks together. On a fresh station, open an App-Builder **terminal** and run:

```bash
cd /data/aip && git clone https://github.com/edge-ai/AIPV1EMI0001.git
```

If that directory already exists, don’t overwrite it; confirm with the station administrator that it is your tutorial checkout. In the file browser open `AIPV1EMI0001/notebooks/01-build-yolov7.ipynb`. The next two cells select the working directory and check for the station tools. Leave cleanup off unless the station is reserved for the live test.

In [ ]:
import os
import sys
from pathlib import Path

FLAVOR = "yolov7"
AIP_ID = "AIPV1EMI0001"
WORKSPACE_MIB = 1024
CLEANUP_EAIS_RESOURCES = False

# The notebook lives in notebooks/, but every command runs from the AI Processor root.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
assert Path.cwd() == Path("/data/aip") / AIP_ID, f"Open this notebook from /data/aip/{AIP_ID}/notebooks"

sys.path.insert(0, "tools")
import flavors  # reads flavors/<FLAVOR>/flavor.json, the single source of truth for this model

flavor = flavors.load(Path.cwd(), FLAVOR)
MODEL_DIR = f"models/{AIP_ID}_Yolo_Detector"
os.environ.update(
    AIP_ID=AIP_ID, FLAVOR=FLAVOR, WORKSPACE_MIB=str(WORKSPACE_MIB), MODEL_DIR=MODEL_DIR,
    ENGINE=f"{MODEL_DIR}/1/{flavor['engine']}",
    TRTEXEC=os.environ.get("TRTEXEC_BIN", "/usr/src/tensorrt/bin/trtexec"),
    MS3="http://ms3_videostreaming:8000",
    CLEANUP_EAIS_RESOURCES=str(int(CLEANUP_EAIS_RESOURCES)),
)
print(f"{flavor['display_name']}: ONNX {flavor['onnx']}, engine {flavor['engine']}, WORKSPACE_MIB={WORKSPACE_MIB}")

In [ ]:
%%bash
set -euo pipefail
for tool in eais-tools make jq curl timeout git "$TRTEXEC"; do
    command -v "$tool" > /dev/null || { echo "missing: $tool"; exit 1; }
done
test -d /opt/nvidia/deepstream/deepstream/sources/includes
head -n 1 /etc/nv_tegra_release
/usr/local/cuda/bin/nvcc --version | grep release
banner=$("$TRTEXEC" --help 2>&1 || true)
[[ $banner =~ TensorRT\ v[0-9]+ ]] && echo "${BASH_REMATCH[0]}"
python3 -c "import torch, onnx, onnxruntime; print('torch', torch.__version__, '| onnx', onnx.__version__, '| onnxruntime', onnxruntime.__version__)"
df -h /data/aip | tail -n 1

## 1. Select YOLOv7-tiny

[`tools/flavors.py`](../tools/flavors.py) sets this checkout’s active detector from the [flavor recipe](../flavors/yolov7/flavor.json). It updates the manifest, Triton and DeepStream configurations, store metadata and `LICENSE.weights`. **Check:** the selected flavor should be `yolov7`. The weights license is GPL-3.0.

In [ ]:
%%bash
set -euo pipefail
python3 tools/flavors.py use "$FLAVOR"
python3 tools/flavors.py check
python3 tools/flavors.py list

## 2. Export the weights and inspect the graph

ONNX is the graph TensorRT will compile. On the first export, the tool downloads `yolov7-tiny.pt` and verifies its checksum. It checks the graph’s input and outputs even when reusing an existing ONNX file. No training is needed.

The exporter checks out a pinned revision of the YOLOv7 repository. Use the station’s PyTorch, ONNX and ONNX Runtime packages; there is no Ultralytics install step.

In [ ]:
%%bash
set -euo pipefail
python3 tools/export_model.py "$FLAVOR"

**Check the graph before you compile it.** Look for `images` as the input and `num_dets`, `det_boxes`, `det_scores`, `det_classes` as outputs. The `EfficientNMS_TRT` plugin suppresses overlapping boxes inside the engine; `num_dets` tells the parser how many of the 100 slots to read.

In [ ]:
import onnx

model = onnx.load(str(flavors.onnx_path(Path.cwd(), flavor)), load_external_data=False)


def describe(value):
    dims = [d.dim_param or d.dim_value or "?" for d in value.type.tensor_type.shape.dim]
    return f"{value.name}: {onnx.TensorProto.DataType.Name(value.type.tensor_type.elem_type)} {dims}"


print("inputs: ", [describe(v) for v in model.graph.input])
print("outputs:", [describe(v) for v in model.graph.output])
print("operators:", len(model.graph.node), "| TensorRT plugin nodes:",
      sorted({node.op_type for node in model.graph.node if node.op_type.startswith("EfficientNMS")}) or "none")

## 3. Build the parser and engine

First compile the C++ parser with `SKIP_ENGINE=1`. DeepStream’s `nvinferserver` calls `NvDsInferParseCustomEfficientNMS` to translate this model’s output layout into detection boxes. It maps COCO car (2), person (0) and truck (7) to the AIP’s label order: car (0), person (1), truck (2). The command should list `libnvds_infercustomparser.so`.

In [ ]:
%%bash
set -euo pipefail
SKIP_ENGINE=1 bin/build.sh "$FLAVOR"
ls -l "$MODEL_DIR/libnvds_infercustomparser.so"

Now run the full build. `trtexec` compiles the graph for **this GPU** at 640 × 640 and permits FP16 kernels. The engine accepts batch sizes 1–8 and is tuned for 8; this is **not** a promise of eight real-time cameras.

This is the slow step: allow tens of minutes. The command should list the selected `.engine` file. If it fails, look in `tmp/engines/<flavor>/build.log`. Keep the 1024 MiB builder workspace on this Jetson; the CPU and GPU share memory.

In [ ]:
%%bash
set -euo pipefail
bin/build.sh "$FLAVOR"
ls -lh "$ENGINE"

## 4. Check that the engine runs

Load the engine and run a batch of one. Look for `OK: the engine loads and runs on this station`. This uses synthetic input: it can catch an incompatible engine, but tells you nothing about detection quality.

In [ ]:
%%bash
set -euo pipefail
mkdir -p tmp/build
"$TRTEXEC" --loadEngine="$ENGINE" --shapes=images:1x3x640x640 --duration=1 --warmUp=200 --iterations=1 \
    > tmp/build/smoke-test.log 2>&1 || { tail -n 40 tmp/build/smoke-test.log; exit 1; }
grep -q '&&&& PASSED' tmp/build/smoke-test.log
echo "OK: the engine loads and runs on this station"

## 5. Check the package configuration

`eais-tools aip check` checks the AIP layout. The Python cell checks that Triton and DeepStream can read the model settings and the NvDCF tracker configuration has its required sections. Look for `OK: Triton, DeepStream and NvDCF configurations` before packaging.

In [ ]:
%%bash
set -euo pipefail
eais-tools aip check --uuid "$AIP_ID"

In [ ]:
import json
import yaml
from eais_tools.ds_config_validator import validate_ds_config
from eais_tools.services.configs_service import validate_model_config
from eais_tools.triton_config_validator import validate_specified_triton_config

manifest = json.loads(Path("manifest.json").read_text())
detector, tracker = manifest["primary_processor"], manifest["tracker_engine"]
assert detector["config_plan"] == flavor["infer_config"], "run tools/flavors.py use for this flavor"
pbtxt = str(Path(MODEL_DIR, "config.pbtxt").resolve())
triton = validate_specified_triton_config(pbtxt)
assert triton.result and validate_model_config(detector["name"], pbtxt), triton.error_message
ds = validate_ds_config(str(Path("configs", detector["config_plan"]).resolve()))
assert ds.success, ds.message
nvdcf = yaml.safe_load(Path("configs", tracker["config_plan"]).read_text().removeprefix("%YAML:1.0"))
assert {"BaseConfig", "TargetManagement", "StateEstimator", "VisualTracker"} <= nvdcf.keys()
assert Path(tracker["lib_path"]).is_file(), tracker["lib_path"]
print("OK: Triton, DeepStream and NvDCF configurations")

## 6. Create the installable archive

EAIS Tools assembles the runtime files in `dist/`. The [`.include`](../.include) list adds the licenses, manual, Node-RED flow and dashboard. Export writes **`AIPV1EMI0001_dist.tgz`** and the cell prints its checksum.

**If you only need the archive, stop here.** The remaining cells register this AIP and start a test pipeline on the station; run them only on a development station reserved for the exercise.

In [ ]:
%%bash
set -euo pipefail
eais-tools aip build --uuid "$AIP_ID" --include-file .include

In [ ]:
%%bash
set -euo pipefail
rm -f "${AIP_ID}_dist.tgz"
eais-tools aip export --uuid "$AIP_ID"
sha256sum "${AIP_ID}_dist.tgz"

## 7. Register this detector for a live test

The station can scan the AIP directly from `/data/aip`. This request targets only `AIPV1EMI0001` and does not force an update. Check that the registered version **and detector** match the manifest: different flavors share an ID and version. If the scan or check fails, stop and consult the administrator; don’t force a rescan.

In [ ]:
%%bash
set -euo pipefail
if ! curl -fsS -X POST "$MS3/aiprocessor/scan/$AIP_ID?force=false" > /dev/null; then
    echo "Targeted scan failed. If this AIP is registered with ms7, coordinate with the station administrator." >&2
    exit 1
fi
AIP=$(curl -fsS "$MS3/aiprocessors" | jq -ec --arg id "$AIP_ID" '.[] | select(.unique_id == $id)')
test "$(jq -r .version <<< "$AIP")" = "$(jq -r .version manifest.json)"
expected=$(jq -er .primary_processor.model manifest.json)
jq -e --arg model "$expected" '.primary_processor.model == $model' <<< "$AIP" > /dev/null || {
    echo "Registered detector does not match the selected flavor; stop before testing." >&2; exit 1; }
echo "OK: ms3 lists $AIP_ID v$(jq -r .version <<< "$AIP") (pk $(jq -r .pk <<< "$AIP"))"

## 8. Run one sample video

`eais-tools pipeline test` starts a headless, one-input pipeline with inference, NvDCF tracking and the callback. Let it collect messages for 90 seconds. A timeout is expected; **the next cell**, not the timeout, decides whether the test passed.

The test tool can replace `eais_*` resources. The cell refuses to start if any already exist. Don’t run another test concurrently. `--retain_resources` leaves the test pipeline on the station for inspection.

In [ ]:
%%bash
set -euo pipefail
eais_resources() { curl -fsS "$MS3/$1" | jq '[.[] | select(.name | startswith("eais_"))] | length'; }
[ "$(eais_resources pipelines)" = 0 ] && [ "$(eais_resources medias)" = 0 ] || {
    echo "eais_* pipelines or media already exist: another test owns them"; exit 1; }
mkdir -p tmp && rm -f tmp/callback_dump.json
status=0
COLUMNS=160 timeout --signal=INT --kill-after=30 90 eais-tools pipeline test --aip "$AIP_ID" \
    --inputs_num 1 --callback_output --retain_resources --dump_file tmp/callback_dump.json || status=$?
[ "$status" = 0 ] || [ "$status" = 124 ]

**Check for a detection.** Read the messages from [`callback.py`](../callback.py) collected in `tmp/callback_dump.json`. The [callback contract](../doc/USER_MANUAL.md#callback-output-metadata-contract) distinguishes an unavailable input (no observation) from a frame with zero objects. At least one observed frame must contribute a positive count. On success you’ll see peak counts per class and the test pipeline ID; this is not a labelled accuracy test.

In [ ]:
import json
import requests

text, pos, records, decoder = Path("tmp/callback_dump.json").read_text(), 0, [], json.JSONDecoder()
while pos < len(text):  # the dump is a sequence of JSON arrays, one per message
    if text[pos].isspace():
        pos += 1
        continue
    batch, pos = decoder.raw_decode(text, pos)
    records.extend(batch)
assert records, "no callback messages received"

peak = {c: 0 for c in manifest["primary_processor"]["classes"]}
for record in records:
    assert {"since", "until", "data"} <= record.keys(), record.keys()
    for source in record["data"].values():
        assert source["obj_counter"].keys() == peak.keys(), source["obj_counter"].keys()
        assert {"objInROIcnt", "objLCCurrCnt"} <= source["analytics_obj_cnt"].keys()
        if not source["available"]:  # no frames in this window; counts are null, not zero
            continue
        for c in peak:
            peak[c] = max(peak[c], source["obj_counter"][c]["max_val"])
print(f"{len(records)} callback records, peak objects per frame: {peak}")
assert any(peak.values()), "no objects detected"

pipelines = requests.get(f"{os.environ['MS3']}/pipelines", timeout=30).json()
(PIPELINE_ID,) = [p["pk"] for p in pipelines if p["name"] == "eais_pipeline"]
os.environ["PIPELINE_ID"] = PIPELINE_ID
print(f"OK: test pipeline {PIPELINE_ID} delivers detections and valid metadata")

## 9. Finish the exercise

By default the archive and test resources are kept. Stop your test pipeline in the station UI when you’re done; remove its test pipeline and media before repeating the exercise.

On a **reserved** station only, setting `CLEANUP_EAIS_RESOURCES = True` in setup enables `eais-tools pipeline clean` below. That command deletes **all** `eais_*` pipelines and media. The guard checks the pipeline ID and number of media; it cannot prove media ownership. If anything is unfamiliar, leave it off.

To display the counts, follow the [dashboard setup](../logic/README.md). To assess detection quality or camera capacity, continue to the [model comparison](04-compare-models.ipynb).

In [ ]:
%%bash
set -euo pipefail
if [ "${CLEANUP_EAIS_RESOURCES:-0}" = 1 ]; then
    curl -fsS "$MS3/pipelines" | jq -e --arg id "$PIPELINE_ID" \
        '[.[] | select(.name | startswith("eais_")) | .pk] == [$id]' > /dev/null
    curl -fsS "$MS3/medias" | jq -e '[.[] | select(.name | startswith("eais_"))] | length <= 1' > /dev/null
    eais-tools pipeline clean
    curl -fsS "$MS3/pipelines" | jq -e --arg id "$PIPELINE_ID" 'all(.[]; .pk != $id)' > /dev/null
else
    echo "Skipped station-wide pipeline cleanup; set CLEANUP_EAIS_RESOURCES=True only on a reserved station."
fi
sha256sum "${AIP_ID}_dist.tgz"